In [1]:
from datetime import datetime
import pandas as pd

_tdy = datetime.today().strftime("%Y-%m-%d")

out = pd.read_csv(f"../results/{_tdy}/all_matchups_fg3_predictions.csv")


In [2]:
from __future__ import annotations

import numpy as np
import pandas as pd


# ============================================================
# THREES TICKET SELECTORS
# expects output from predict_game_fg3 (and optionally add_prob_ge_k)
# ============================================================
# Required columns (raw preds):
#   player, team, opp, is_home, pred_fg3a, pred_rate, pred_fg3, baseline_fg3, delta_fg3
# If you ran add_prob_ge_k(out, k=2) / add_prob_ge_k(out, k=3), you also have:
#   p_ge_2, p_ge_3
# ============================================================

def _require_cols(df: pd.DataFrame, req: set[str]) -> None:
    missing = sorted(req - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def add_matchup_key(df: pd.DataFrame) -> pd.DataFrame:
    """Adds matchup_key as AWAY@HOME."""
    out = df.copy()
    out["matchup_key"] = np.where(
        out["is_home"].astype(int) == 1,
        out["opp"].astype(str) + "@" + out["team"].astype(str),   # away@home
        out["team"].astype(str) + "@" + out["opp"].astype(str),   # away@home
    )
    return out


# ------------------------------------------------------------
# 1) "2+" ticket (like your old select_2plus_ticket)
# ------------------------------------------------------------
def select_2plus_ticket(
    df: pd.DataFrame,
    *,
    n_legs: int = 10,
    min_pred_fg3a: float = 5.8,
    min_p_ge_2: float = 0.62,
    min_p_ge_3: float = 0.30,
    max_per_team: int = 3,
    rank_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Picks n legs for 2+ threes.
    Filters:
      pred_fg3a >= min_pred_fg3a
      p_ge_2   >= min_p_ge_2
      p_ge_3   >= min_p_ge_3 (optional quality gate)
    Ranks (default): p_ge_2, pred_fg3, pred_fg3a
    """
    req = {"player","team","opp","is_home","pred_fg3a","pred_fg3","p_ge_2","p_ge_3"}
    _require_cols(df, req)

    out = add_matchup_key(df)

    pool = out[
        (out["pred_fg3a"] >= float(min_pred_fg3a)) &
        (out["p_ge_2"] >= float(min_p_ge_2)) &
        (out["p_ge_3"] >= float(min_p_ge_3))
    ].copy()

    if pool.empty:
        return pool

    if rank_cols is None:
        rank_cols = ["p_ge_2", "pred_fg3", "pred_fg3a"]

    pool = pool.sort_values(rank_cols, ascending=[False] * len(rank_cols))

    if max_per_team is not None:
        pool["_team_rank"] = pool.groupby("team").cumcount()
        pool = pool[pool["_team_rank"] < int(max_per_team)].copy()
        pool.drop(columns=["_team_rank"], inplace=True)

    return pool.head(int(n_legs)).reset_index(drop=True)


# ------------------------------------------------------------
# 2) "Jackpot" ticket (+k over baseline)
# ------------------------------------------------------------
def select_jackpot_threes_ticket(
    df: pd.DataFrame,
    *,
    n_legs: int = 3,
    over_baseline_col: str = "p_over_baseline_2",
    min_pred_fg3a: float = 5.5,
    min_p_over_baseline: float = 0.18,
    min_delta_fg3: float = 0.75,
    max_per_team: int = 1,
) -> pd.DataFrame:
    """
    Jackpot = players projected meaningfully above baseline threes.

    Requires columns:
      player, team, opp, is_home, pred_fg3a, pred_fg3, baseline_fg3, delta_fg3, over_baseline_col
    """
    req = {"player","team","opp","is_home","pred_fg3a","pred_fg3","baseline_fg3","delta_fg3", over_baseline_col}
    _require_cols(df, req)

    out = add_matchup_key(df)

    pool = out[
        (out["pred_fg3a"] >= float(min_pred_fg3a)) &
        (out["delta_fg3"] >= float(min_delta_fg3)) &
        (out[over_baseline_col] >= float(min_p_over_baseline))
    ].copy()

    if pool.empty:
        return pool

    pool = pool.sort_values(
        [over_baseline_col, "delta_fg3", "pred_fg3", "pred_fg3a"],
        ascending=[False, False, False, False],
    )

    if max_per_team is not None:
        pool["_team_rank"] = pool.groupby("team").cumcount()
        pool = pool[pool["_team_rank"] < int(max_per_team)].copy()
        pool.drop(columns=["_team_rank"], inplace=True)

    return pool.head(int(n_legs)).reset_index(drop=True)


# ------------------------------------------------------------
# 3) Matchup coverage ticket (2 per matchup + insurance)
# ------------------------------------------------------------
def select_matchup_coverage_threes_ticket(
    df: pd.DataFrame,
    *,
    players_per_matchup: int = 2,
    insurance_per_matchup: int = 1,   # set 0 for "exactly N"
    min_pred_fg3a: float = 5.6,
    min_p_ge_2: float = 0.60,
    min_p_ge_3: float = 0.30,
    max_legs: int | None = None,
) -> pd.DataFrame:
    """
    Matchup-coverage selector (THREES):
      - Groups rows into matchups (AWAY@HOME)
      - Picks top players_per_matchup + insurance_per_matchup per matchup
      - Ranks by: p_ge_2, then pred_fg3, then pred_fg3a

    Requires columns:
      player, team, opp, is_home, pred_fg3a, pred_fg3, p_ge_2, p_ge_3
    """
    req = {"player","team","opp","is_home","pred_fg3a","pred_fg3","p_ge_2","p_ge_3"}
    _require_cols(df, req)

    out = add_matchup_key(df)

    pool = out[
        (out["pred_fg3a"] >= float(min_pred_fg3a)) &
        (out["p_ge_2"] >= float(min_p_ge_2)) &
        (out["p_ge_3"] >= float(min_p_ge_3))
    ].copy()

    if pool.empty:
        return pool

    pool = pool.sort_values(
        ["matchup_key", "p_ge_2", "pred_fg3", "pred_fg3a"],
        ascending=[True, False, False, False],
    )
    pool["_rank"] = pool.groupby("matchup_key").cumcount()

    keep = int(players_per_matchup) + int(insurance_per_matchup)
    ticket = pool[pool["_rank"] < keep].copy()

    if max_legs is not None and len(ticket) > int(max_legs):
        ticket = ticket.sort_values(["p_ge_2","pred_fg3"], ascending=False).head(int(max_legs))

    return (
        ticket
        .drop(columns=["_rank"])
        .sort_values(["matchup_key","p_ge_2","pred_fg3"], ascending=[True, False, False])
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 4) Pencil labels for threes (3+ / 2+ / coverage_only)
# ------------------------------------------------------------
def assign_pencil_decision_threes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds df['pencil'] labels for threes.

    Requires: p_ge_2, p_ge_3, pred_fg3a
    """
    req = {"pred_fg3a","p_ge_2","p_ge_3"}
    _require_cols(df, req)

    out = df.copy()

    conditions = [
        # Strong 3+ candidates
        (out["p_ge_3"] >= 0.35) & (out["pred_fg3a"] >= 6.0),

        # Solid 2+ candidates
        (out["p_ge_2"] >= 0.60) & (out["pred_fg3a"] >= 4.5),
    ]
    choices = ["3+", "2+"]

    out["pencil"] = np.select(conditions, choices, default="coverage_only")
    return out


# ============================================================
# Example usage
# ============================================================
# out_all = pd.read_csv(f"../results/{_tdy}/all_matchups_fg3_predictions.csv")
# # if you didn't already add these:
# # out_all = add_prob_ge_k(out_all, k=2)
# # out_all = add_prob_ge_k(out_all, k=3)
#
# ticket_2p = select_2plus_ticket(out_all, n_legs=10)
# ticket_cov = select_matchup_coverage_threes_ticket(out_all, players_per_matchup=2, insurance_per_matchup=1)
# labeled = assign_pencil_decision_threes(out_all)
#
# print(ticket_2p[["player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]])
# print(ticket_cov[["matchup_key","player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]])
# print(labeled[["player","team","pred_fg3a","p_ge_2","p_ge_3","pencil"]].head(25))


In [3]:

out_all = pd.read_csv(f"../results/{_tdy}/all_matchups_fg3_predictions.csv")
# if you didn't already add these:
# out_all = add_prob_ge_k(out_all, k=2)
# out_all = add_prob_ge_k(out_all, k=3)

ticket_2p = select_2plus_ticket(out_all, n_legs=10)
ticket_cov = select_matchup_coverage_threes_ticket(out_all, players_per_matchup=2, insurance_per_matchup=1)
labeled = assign_pencil_decision_threes(out_all)

display(ticket_2p[["player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]])
display(ticket_cov[["matchup_key","player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]])
display(labeled[["player","team","pred_fg3a","p_ge_2","p_ge_3","pencil"]].head(25))


,player,team,opp,is_home,pred_fg3a,pred_rate,pred_fg3,p_ge_2,p_ge_3
0,A.J. Green,MIL,NOP,0,7.817362,0.333439,2.606616,0.733890,0.483230
1,Nickeil Alexander-Walker,ATL,MIA,1,6.572263,0.335204,2.203052,0.646173,0.378104
2,Donovan Mitchell,CLE,CHA,0,6.454051,0.334076,2.156144,0.634611,0.365505
3,Anthony Edwards,MIN,DAL,1,6.603176,0.326286,2.154525,0.634207,0.365069
4,Jamal Murray,DEN,POR,0,6.338797,0.331590,2.101882,0.620869,0.350877


,matchup_key,player,team,opp,is_home,pred_fg3a,pred_rate,pred_fg3,p_ge_2,p_ge_3
0,CLE@CHA,Donovan Mitchell,CLE,CHA,0,6.454051,0.334076,2.156144,0.634611,0.365505
1,DAL@MIN,Anthony Edwards,MIN,DAL,1,6.603176,0.326286,2.154525,0.634207,0.365069
2,DEN@POR,Jamal Murray,DEN,POR,0,6.338797,0.331590,2.101882,0.620869,0.350877
3,MIA@ATL,Nickeil Alexander-Walker,ATL,MIA,1,6.572263,0.335204,2.203052,0.646173,0.378104
4,MIL@NOP,A.J. Green,MIL,NOP,0,7.817362,0.333439,2.606616,0.733890,0.483230


,player,team,pred_fg3a,p_ge_2,p_ge_3,pencil
0,Donovan Mitchell,CLE,6.454051,0.634611,0.365505,3+
1,James Harden,CLE,5.754018,0.565646,0.295706,coverage_only
2,Sam Merrill,CLE,5.409102,0.539258,0.271268,coverage_only
3,Jaylon Tyson,CLE,3.220164,0.285735,0.091362,coverage_only
4,Keon Ellis,CLE,3.009862,0.265568,0.080966,coverage_only
5,Craig Porter Jr.,CLE,1.163986,0.058590,0.007342,coverage_only
6,Jarrett Allen,CLE,0.189729,0.001919,0.000040,coverage_only
7,Andrew Nembhard,IND,5.710635,0.567429,0.297400,coverage_only
8,Pascal Siakam,IND,4.735966,0.457214,0.202602,coverage_only
9,Bub Carrington,WAS,4.661705,0.452923,0.199301,coverage_only


In [4]:
display(labeled[labeled['pencil'] != 'coverage_only'][["player","team","pred_fg3a","p_ge_2","p_ge_3","pencil"]].head(25))


,player,team,pred_fg3a,p_ge_2,p_ge_3,pencil
0,Donovan Mitchell,CLE,6.454051,0.634611,0.365505,3+
41,Nickeil Alexander-Walker,ATL,6.572263,0.646173,0.378104,3+
57,Anthony Edwards,MIN,6.603176,0.634207,0.365069,3+
74,A.J. Green,MIL,7.817362,0.733890,0.483230,3+
102,Jamal Murray,DEN,6.338797,0.620869,0.350877,3+
